# 差分メモ

- 対象: `[2]dpc-starter-train-v1.ipynb`
- 元: `dpc-starter-train-cv5.ipynb`
- 種別: 改修（学習用: full-train）
- 変更点:
  - CVを廃止し、全trainデータで学習
  - `akk→en` + `en→akk` の prefix multi-task で2倍化
  - 学習モデルを `google/byt5-base` に変更
- 変更理由/仮説:
  - タスク方向を増やすことで表現学習を強化し、本タスク（akk→en）の汎化を狙う
  - 途中checkpoint保存を無効化（ディスク枯渇対策）
  - CUDA OOM対策: batch=1, grad_accum増, gradient checkpointing, PYTORCH_CUDA_ALLOC_CONF


# Deep Past Initiative – Machine Translation (Training Notebook)

This notebook is a **starter / baseline** for this Kaggle competition.

Main ideas:
- Use **ByT5** to handle noisy Akkadian transliterations at the character level
- Perform **simple sentence alignment** to increase training data
- Fine-tune using HuggingFace `Trainer`


Inference Code is [here](https://www.kaggle.com/code/takamichitoda/dpc-starter-infer).

In [1]:
# (Optional) Kaggle environment usually includes required libs.
# If your environment is missing something, uncomment and install as needed.
# !pip install -q datasets transformers accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.5 MB/s eta 0:00:00


In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc
import re
import math
import unicodedata
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

2026-02-28 01:18:54.092481: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772241534.416066      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772241534.508963      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772241535.319972      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772241535.320025      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772241535.320028      24 computation_placer.cc:177] computation placer alr

In [3]:
class Config:
    # Akkadian transliteration contains a lot of noise and many unknown words, so
    # ByT5, which processes text at the character (byte) level rather than the word level, is a strong choice.
    MODEL_NAME = "google/byt5-base"

    # ByT5 tends to produce longer token sequences, but 512 tokens is enough at the sentence level.
    MAX_LENGTH = 512

    # Training
    EPOCHS = 20
    LEARNING_RATE = 2e-4

    # ByT5-base is heavier; keep per-device batch small and use grad-accum.
    PER_DEVICE_TRAIN_BATCH_SIZE = 1
    GRADIENT_ACCUMULATION_STEPS = 8

    # Output
    OUTPUT_DIR = "./byt5-akkadian-model/fulltrain_byt5-base_multitask"


In [4]:
# Fix the seed (for reproducibility).
def seed_everything(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
    
seed_everything()

In [5]:
INPUT_DIR = "/kaggle/input/competitions/deep-past-initiative-machine-translation"
train_df = pd.read_csv(f"{INPUT_DIR}/train.csv")
test_df = pd.read_csv(f"{INPUT_DIR}/test.csv")

In [6]:
print(f"Original Train Data: {len(train_df)} docs")

Original Train Data: 1561 docs


In [7]:
def simple_sentence_aligner(df):
    """
    【戦略の肝】
    Trainデータの「文書(複数文)」を、Testデータと同じ「文(1文)」に分割します。
    ここでは「英語の文数」と「アッカド語の行数」が一致する場合のみ分割する
    というヒューリスティック（簡易ルール）を使います。

    NOTE:
      - Sentence splitting increases samples per original document.
      - To avoid leakage in CV, we keep `oare_id` as a grouping key.
    """
    aligned_data = []

    for _, row in df.iterrows():
        oare_id = row["oare_id"]
        src = str(row["transliteration"])
        tgt = str(row["translation"])

        # Split the English text by sentence-ending punctuation.
        tgt_sents = [t.strip() for t in re.split(r"(?<=[.!?])\s+", tgt) if t.strip()]

        # Assume the Akkadian text is often separated by newlines and split accordingly.
        src_lines = [s.strip() for s in src.split('\n') if s.strip()]

        # If the counts match, trust it as 1-to-1 pairs and use the split version.
        if len(tgt_sents) > 1 and len(tgt_sents) == len(src_lines):
            for s, t in zip(src_lines, tgt_sents):
                if len(s) > 3 and len(t) > 3:  # Remove junk/noisy data.
                    aligned_data.append({"oare_id": oare_id, "transliteration": s, "translation": t})
        else:
            # If splitting fails (counts don't match), keep the original document pair as-is (safe fallback).
            aligned_data.append({"oare_id": oare_id, "transliteration": src, "translation": tgt})

    return pd.DataFrame(aligned_data, columns=["oare_id", "transliteration", "translation"])


In [8]:
# Run data augmentation.
train_expanded = simple_sentence_aligner(train_df)
print(f"Expanded Train Data: {len(train_expanded)} sentences (Alignment applied)")

train_expanded = train_expanded.reset_index(drop=True)
assert set(["oare_id", "transliteration", "translation"]).issubset(train_expanded.columns)

# ------------------------------------------
# Prefix multi-task: akk→en + en→akk (2x)
# ------------------------------------------
multitask_df = pd.concat(
    [
        train_expanded.assign(
            task="akk2en",
            source=train_expanded["transliteration"],
            target=train_expanded["translation"],
        ),
        train_expanded.assign(
            task="en2akk",
            source=train_expanded["translation"],
            target=train_expanded["transliteration"],
        ),
    ],
    ignore_index=True,
)[["oare_id", "task", "source", "target"]]

print(f"Multi-task Train Data: {len(multitask_df)} samples (2x directions)")
# Deterministic shuffle so batches mix directions
multitask_df = multitask_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(multitask_df.head())


Expanded Train Data: 1561 sentences (Alignment applied)
Prepared 5 folds (GroupKFold by oare_id)


In [9]:
# ==========================================
# 3. Tokenization & preprocessing
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)

PREFIX_AKK_EN = "translate Akkadian to English: "
PREFIX_EN_AKK = "translate English to Akkadian: "

_SUBSCRIPT_NUM_MAP = str.maketrans({
    "₀": "0", "₁": "1", "₂": "2", "₃": "3", "₄": "4",
    "₅": "5", "₆": "6", "₇": "7", "₈": "8", "₉": "9",
})


def normalize_transliteration(text):
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(_SUBSCRIPT_NUM_MAP)

    # Unify common gap markers to a single token for better pattern consistency.
    text = text.replace("…", " <gap> ")
    text = re.sub(r"\b[xX]{1,}\b", " <gap> ", text)

    # Collapse whitespace noise introduced by normalization/replacements.
    text = re.sub(r"\s+", " ", text).strip()
    return text


def preprocess_function(examples):
    tasks = examples["task"]
    sources = examples["source"]
    targets = examples["target"]

    inputs = []
    out_targets = []

    for t, s, y in zip(tasks, sources, targets):
        if t == "akk2en":
            s_norm = normalize_transliteration(s)
            inputs.append(PREFIX_AKK_EN + s_norm)
            out_targets.append(str(y))
        elif t == "en2akk":
            inputs.append(PREFIX_EN_AKK + str(s))
            out_targets.append(normalize_transliteration(y))
        else:
            raise ValueError(f"Unknown task: {t}")

    model_inputs = tokenizer(inputs, max_length=Config.MAX_LENGTH, truncation=True)
    labels = tokenizer(out_targets, max_length=Config.MAX_LENGTH, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [10]:
# ==========================================
# 4. Full-data training (fine-tuning)
# ==========================================

# Build a single HF dataset (full train, no CV).
ds_train = Dataset.from_pandas(multitask_df)

# Tokenize
train_tokenized = ds_train.map(
    preprocess_function,
    batched=True,
    remove_columns=ds_train.column_names,
)

# Model
model = AutoModelForSeq2SeqLM.from_pretrained(Config.MODEL_NAME)
model.config.use_cache = False
model.gradient_checkpointing_enable()
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

os.makedirs(Config.OUTPUT_DIR, exist_ok=True)

args = Seq2SeqTrainingArguments(
    output_dir=Config.OUTPUT_DIR,

    # No CV / no validation in this notebook
    eval_strategy="no",

    save_strategy="no",

    # Checkpoints (incl. optimizer states) can be huge; save only final model at the end.

    learning_rate=Config.LEARNING_RATE,
    fp16=False,

    # Memory savers
    gradient_checkpointing=True,
    group_by_length=True,

    per_device_train_batch_size=Config.PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=Config.GRADIENT_ACCUMULATION_STEPS,

    weight_decay=0.01,
    num_train_epochs=Config.EPOCHS,

    predict_with_generate=False,

    logging_strategy="steps",
    logging_steps=200,
    disable_tqdm=True,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tokenized,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Starting Training (full train, multi-task, FP32 mode)...")
trainer.train()

trainer.save_model(Config.OUTPUT_DIR)
tokenizer.save_pretrained(Config.OUTPUT_DIR)
print(f"Saved model to: {Config.OUTPUT_DIR}")


FOLD 0/4


Map:   0%|          | 0/1248 [00:00<?, ? examples/s]

Map:   0%|          | 0/313 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

...

Starting Training (fold CV, FP32 mode)...
{'loss': 1.4221, 'grad_norm': 1.6517127752304077, 'learning_rate': 0.00019006369426751593, 'epoch': 1.0}
{'eval_loss': 0.7940374612808228, 'eval_chrf': 3.5977744083612984, 'eval_bleu': 5.356709509693513e-08, 'eval_geo_mean': 0.0004390015078220224, 'eval_runtime': 55.2484, 'eval_samples_per_second': 5.647, 'eval_steps_per_second': 1.412, 'epoch': 1.0}
{'loss': 0.8854, 'grad_norm': 1.072219729423523, 'learning_rate': 0.00018006369426751593, 'epoch': 2.0}
{'eval_loss': 0.686783492565155, 'eval_chrf': 4.135569794730734, 'eval_bleu': 2.7558154618717862e-06, 'eval_geo_mean': 0.0033759246413344133, 'eval_runtime': 55.278, 'eval_samples_per_second': 5.644, 'eval_steps_per_second': 1.411, 'epoch': 2.0}
{'loss': 0.7717, 'grad_norm': 1.0232129096984863, 'learning_rate': 0.00017006369426751593, 'epoch': 3.0}
{'eval_loss': 0.6358160376548767, 'eval_chrf': 4.404059035624224, 'eval_bleu': 3.9100425678806845e-06, 'eval_geo_mean': 0.004149705808939987, 'eval_ru

Map:   0%|          | 0/1249 [00:00<?, ? examples/s]

Map:   0%|          | 0/312 [00:00<?, ? examples/s]

/tmp/ipykernel_24/184818429.py:92: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Starting Training (fold CV, FP32 mode)...
{'loss': 1.3967, 'grad_norm': 1.5285295248031616, 'learning_rate': 0.00019006369426751593, 'epoch': 1.0}
{'eval_loss': 0.771103024482727, 'eval_chrf': 3.8379602922307807, 'eval_bleu': 8.07905195914795e-07, 'eval_geo_mean': 0.0017608827507270075, 'eval_runtime': 55.3689, 'eval_samples_per_second': 5.635, 'eval_steps_per_second': 1.409, 'epoch': 1.0}
{'loss': 0.8612, 'grad_norm': 1.2063767910003662, 'learning_rate': 0.00018006369426751593, 'epoch': 2.0}
{'eval_loss': 0.6790016293525696, 'eval_chrf': 4.229632134463456, 'eval_bleu': 2.582999818716742e-06, 'eval_geo_mean': 0.0033053198085143914, 'eval_runtime': 55.3999, 'eval_samples_per_second': 5.632, 'eval_steps_per_second': 1.408, 'epoch': 2.0}
{'loss': 0.7604, 'grad_norm': 1.4247463941574097, 'learning_rate': 0.00017006369426751593, 'epoch': 3.0}
{'eval_loss': 0.6406375169754028, 'eval_chrf': 4.2484149179407975, 'eval_bleu': 4.7484079905996575e-06, 'eval_geo_mean': 0.00449145937794531, 'eval_ru

In [11]:
# --- Notes ---
# This notebook trains a single model using all training data (no CV).
# Output:
#   ./byt5-akkadian-model/fulltrain_byt5-base_multitask/
